# Lesson 04 — Track ID Persistence and the Re-ID Problem

```python
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Demonstrating ID consistency across occlusion
# The key challenge: when an object disappears and reappears, does it keep its ID?

class PersistentTracker:
    def __init__(self, max_disappeared=30, max_distance=100):
        self.next_id     = 0
        self.objects     = {}
        self.disappeared = {}
        self.history     = {}    # id -> list of centroids (for trail drawing)
        self.max_gone    = max_disappeared
        self.max_dist    = max_distance

    def update(self, centroids):
        if not centroids:
            for oid in list(self.disappeared):
                self.disappeared[oid] += 1
                if self.disappeared[oid] > self.max_gone:
                    del self.objects[oid]
                    del self.disappeared[oid]
            return self.objects, self.history

        if not self.objects:
            for c in centroids:
                self.objects[self.next_id]     = np.array(c)
                self.disappeared[self.next_id] = 0
                self.history[self.next_id]     = [c]
                self.next_id += 1
            return self.objects, self.history

        obj_ids   = list(self.objects.keys())
        obj_cents = np.array(list(self.objects.values()))
        new_cents = np.array(centroids)

        from scipy.spatial.distance import cdist
        D    = cdist(obj_cents, new_cents)
        rows = D.min(axis=1).argsort()
        cols = D.argmin(axis=1)[rows]
        used_r, used_c = set(), set()

        for r,c in zip(rows, cols):
            if r in used_r or c in used_c: continue
            if D[r,c] > self.max_dist: continue
            oid = obj_ids[r]
            self.objects[oid]     = new_cents[c]
            self.disappeared[oid] = 0
            self.history.setdefault(oid,[]).append(tuple(new_cents[c]))
            used_r.add(r); used_c.add(c)

        for r in set(range(len(obj_ids)))-used_r:
            self.disappeared[obj_ids[r]] += 1
            if self.disappeared[obj_ids[r]] > self.max_gone:
                del self.objects[obj_ids[r]]
                del self.disappeared[obj_ids[r]]

        for c in set(range(len(centroids)))-used_c:
            self.objects[self.next_id]     = new_cents[c]
            self.disappeared[self.next_id] = 0
            self.history[self.next_id]     = [tuple(new_cents[c])]
            self.next_id += 1

        return self.objects, self.history

print("ID persistence: an object keeps its ID as long as it reappears within max_disappeared frames.")
print("After that, it gets a NEW ID if it comes back (re-identification problem).")
print("Deep learning re-ID (matching appearance) solves this — beyond classical CV scope.")